# A virtual cohort for trial design

This notebook runs a two arm virtual trial on cohorts generated from the durations the
article reports: 150 patients per arm, two years of follow up, and a treatment that cuts the
relapse onset rate to seven tenths of the control rate. It reports the annualised relapse
rate of each arm under three intervals, compares the two arms with a negative binomial rate
ratio, and builds a power curve by repeating the whole trial, which is then put beside the
sample size formula for the same inputs.

## Provenance

**No clinical data are used in this notebook, and nothing in it is a trial protocol.** Both
arms are simulated with the alternating renewal engine, aimed at the two mean durations the
article prints. The treatment effect is imposed by hand and is not an effect the article
reports or a claim about any drug. The sample size formula assumes one fixed analysis of
complete follow up, with no dropout, no adjustment and no interim look, and is here as an
illustration of what the rate, the spread and the length of follow up cost, not as a design
tool.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import msrelapse

# Every random step of this notebook is drawn from this one seed.
SEED = 20130910
# The trial. None of these five numbers is from the article.
RATE_RATIO = 0.7
N_PER_ARM = 150
FOLLOWUP_WEEKS = 104.0
ALPHA_LEVEL = 0.05
N_BOOT = 1000

FOLLOWUP_YEARS = FOLLOWUP_WEEKS / msrelapse.WEEKS_PER_YEAR
TAU_HEALTH = msrelapse.PAPER.tau_health_cohort_weeks.value
TAU_RELAPSE = msrelapse.PAPER.tau_no_health_cohort_weeks.value

## The two arms

One relapse onset costs one remission and one relapse, so the onset rate is
`1 / (tau_health + tau_relapse)`. Cutting it to seven tenths therefore means lengthening the
cycle by that factor with the relapse itself untouched, which is what the treated arm is
aimed at below. The records are weekly, as the records of the study are, and every patient
starts in remission rather than in a relapse, so that the first onset of a record is not
forced to week zero.

In [ ]:
TAU_HEALTH_TREATED = (TAU_HEALTH + TAU_RELAPSE) / RATE_RATIO - TAU_RELAPSE
control_onset = msrelapse.effective_onset_rate(*msrelapse.rates_from_means(TAU_HEALTH, TAU_RELAPSE))
treated_onset = msrelapse.effective_onset_rate(
    *msrelapse.rates_from_means(TAU_HEALTH_TREATED, TAU_RELAPSE)
)

print(f"control remission mean {TAU_HEALTH:.1f} weeks, treated {TAU_HEALTH_TREATED:.1f} weeks")
print(f"onset rates {control_onset:.5f} and {treated_onset:.5f} per week")
print(f"their ratio is {treated_onset / control_onset:.3f}")

In [ ]:
def arm_spec(n, tau_health):
    """Return the description of one trial arm."""
    return msrelapse.CohortSpec(
        n=n,
        tau_health=tau_health,
        tau_relapse=TAU_RELAPSE,
        followup_weeks=FOLLOWUP_WEEKS,
        engine="renewal",
        weekly=True,
        start_state="health",
    )


trial_generator = np.random.default_rng(SEED)
control = msrelapse.generate(arm_spec(N_PER_ARM, TAU_HEALTH), rng=trial_generator)
treated = msrelapse.generate(arm_spec(N_PER_ARM, TAU_HEALTH_TREATED), rng=trial_generator)
print(control)
print(treated)

## The annualised relapse rate of each arm

The three intervals answer three questions. The exact Poisson interval takes every relapse to
be an independent event at a rate the whole arm shares. The negative binomial interval widens
it by the spread of the per patient counts. The bootstrap resamples patients and keeps each
patient's count and follow up together, which is the one to read when the follow up is
unequal. These arms are generated from a single shared rate, so the three come out close
together; a real cohort spreads them apart.

In [ ]:
# One generator through the loop, so that the bootstrap of the two arms does not
# resample both of them on the same table of patient indices.
bootstrap_generator = np.random.default_rng(SEED)
arr_rows = []
for name, cohort in (("control", control), ("treated", treated)):
    for method in ("poisson_exact", "nb", "bootstrap"):
        rate = msrelapse.arr(cohort.events, ci=method, n_boot=N_BOOT, rng=bootstrap_generator)
        arr_rows.append(
            (
                name,
                method,
                rate.n_patients,
                rate.n_relapses,
                rate.patient_years,
                rate.arr,
                rate.ci_low,
                rate.ci_high,
            )
        )

rates = pd.DataFrame(
    arr_rows,
    columns=["arm", "ci", "n_patients", "n_relapses", "patient_years", "arr", "ci_low", "ci_high"],
)
rates

In [ ]:
control_arr = msrelapse.arr(control.events)
comparison = msrelapse.compare_arr(control.events, None, treated.events, None, model="nb")

print(f"rate ratio {comparison.rate_ratio:.3f}")
print(f"95% interval {comparison.ci_low:.3f} to {comparison.ci_high:.3f}")
print(f"p value {comparison.p_value:.4g}, fitted dispersion {comparison.dispersion:.4f}")
print(f"arm rates {comparison.arr_a:.3f} and {comparison.arr_b:.3f} per year")

## The power curve

The whole trial is repeated 200 times at each of four sizes, and the share of repeats that
reject at the 5 percent level is the power. One generator runs through every repeat, so the
curve is reproducible from the seed above and no two arms and no two trials share a draw. The
sample size formula is drawn beside it at two dispersions: the one this comparison fitted,
which is zero because every simulated patient carries the same rate, and the dispersion of a
cohort spread as widely as the gamma mixture of the previous notebook.

In [ ]:
N_GRID = (60, 100, 150, 250)
# Trials behind each point of the curve. The share it measures carries a
# standard error of about three percent here; four times as many trials halves
# it and costs four times as much.
N_TRIALS = 200


def one_trial(n, generator):
    """Return the p value of one two arm trial of n patients per arm."""
    arm_a = msrelapse.generate(arm_spec(n, TAU_HEALTH), rng=generator)
    arm_b = msrelapse.generate(arm_spec(n, TAU_HEALTH_TREATED), rng=generator)
    return msrelapse.compare_arr(arm_a.events, None, arm_b.events, None, model="nb").p_value


power_generator = np.random.default_rng(SEED)
power_rows = []
for n in N_GRID:
    p_values = np.array([one_trial(n, power_generator) for _ in range(N_TRIALS)])
    power_rows.append((n, float(np.mean(p_values < ALPHA_LEVEL))))

power = pd.DataFrame(power_rows, columns=["n_per_arm", "power"])
power

In [ ]:
POWERS = (0.5, 0.6, 0.7, 0.8, 0.9, 0.95)
# The dispersion of a cohort whose rates follow a gamma of shape 1.5, which is
# the mixture of the counts notebook.
SPREAD_DISPERSION = 1.0 / 1.5

formula_rows = []
for target in POWERS:
    for dispersion in (comparison.dispersion, SPREAD_DISPERSION):
        formula_rows.append(
            (
                dispersion,
                msrelapse.sample_size_arr(
                    control_arr.arr,
                    RATE_RATIO,
                    dispersion,
                    FOLLOWUP_YEARS,
                    power=target,
                    alpha=ALPHA_LEVEL,
                ),
                target,
            )
        )

formula = pd.DataFrame(formula_rows, columns=["dispersion", "n_per_arm", "power"])
formula

In [ ]:
panel = plt.subplots(figsize=(7.0, 4.2), layout="constrained")[1]
panel.plot(
    power["n_per_arm"],
    power["power"],
    marker="o",
    color="black",
    label=f"{N_TRIALS} simulated trials",
)
for dispersion, block in formula.groupby("dispersion"):
    panel.plot(
        block["n_per_arm"],
        block["power"],
        marker="s",
        linestyle="--",
        label=f"sample_size_arr, dispersion {dispersion:.2f}",
    )
panel.axhline(0.8, linestyle=":", color="grey", label="80 percent power")
panel.set_xlim(0, 400)
panel.set_xlabel("patients per arm")
panel.set_ylabel(f"share of trials with p < {ALPHA_LEVEL}")
panel.legend(fontsize="small")
plt.show()

## Which engine to use

Both engines in this package produce the same kind of record, and they answer different
questions.

The renewal engine draws durations straight from their target means. It has no potential
behind it, it costs almost nothing, and it is the right engine for a trial calculation, for a
null model of what a cohort of a given size can resolve, and for anything where the durations
themselves are the assumption being made.

The mechanistic engine calibrates a double well to the same pair of means and integrates the
stochastic equation. It costs far more, and what it buys is a mechanism: the durations are
then a consequence of a barrier, a noise and a shape rather than an assumption, so a question
about what would happen if the barrier moved, if the two wells were tilted differently, or if
the noise rose, has somewhere to be asked. It is also the engine that shows where the
phenomenological picture breaks down, as the exit time notebook does when it finds the
relapse well of the calibrated potential too shallow to give exponential durations at all.

Use the renewal engine to size a trial, and the mechanistic engine to ask why the record
looks the way it does.

## How to cite

This notebook reproduces the article below, and the package it uses carries the same
reference on every result object. The text is what `msrelapse.citation()` prints.

```text
Bordi I, Umeton R, Ricigliano VAG, Annibali V, Mechelli R, Ristori G, Grassi F, Salvetti M, Sutera A. A mechanistic, stochastic model helps understand multiple sclerosis course and pathogenesis. International Journal of Genomics. 2013;2013:910321. doi:10.1155/2013/910321.

@article{bordi2013mechanistic,
  author    = {Bordi, Isabella and Umeton, Renato and Ricigliano, Vito A. G. and Annibali, Viviana and Mechelli, Rosella and Ristori, Giovanni and Grassi, Francesca and Salvetti, Marco and Sutera, Alfonso},
  title     = {A mechanistic, stochastic model helps understand multiple sclerosis course and pathogenesis},
  journal   = {International Journal of Genomics},
  volume    = {2013},
  pages     = {910321},
  year      = {2013},
  doi       = {10.1155/2013/910321},
  publisher = {Hindawi}
}

% The DOI below is a placeholder: the archive DOI is minted at the first release.
@software{msrelapse,
  author  = {Umeton, Renato},
  title   = {msrelapse: a reference implementation of the Bordi et al. 2013 double well model of multiple sclerosis},
  year    = {2026},
  version = {0.1.0},
  url     = {https://github.com/renato-umeton/multiple-sclerosis-modeling-bordi},
  doi     = {10.5281/zenodo.XXXXXXX}
}
```